## Data Collection

Collect order book and trade data for BTCUSDT from the Binance API.

Data is sampled every second and saved as Parquet files for later use.  
The actual collection runs as a standalone script; this notebook just shows how it’s used.

### Method

- REST API sampled every ~1 second
- Order book: top 100 bid/ask levels each snapshot
- Trades: latest 1000 each step, filtered using `last_trade_id`

### Timestamps & Latency

- Timestamps taken on the client when the request is made
- Latency estimated as ~100ms (half round-trip)

### Data Handling

- Data stored in buffers before saving
- Trade buffer capped to avoid memory issues
- API calls retried if they fail

### Storage

- Saved as Parquet (snappy)
- Order book and trades saved separately
- Files saved in batches

### Notes

- Used REST for simplicity
- eventually move to websocket

In [ ]:
from microstructure_alpha.utils.logger import setup_logger
import microstructure_alpha.data.dataset as m_alpha_data

import pandas as pd
import numpy as np
from binance.client import Client
import time
from tqdm import tqdm
from pathlib import Path

# Setup Logger

In [ ]:
logger = setup_logger(
    "E:\\Quant_Projects\\microstructure-alpha-engine\\microstructure-alpha-engine\\logs\\data_collection_notebook.log"
)

# Config

In [ ]:
SYMBOL = "BTCUSDT"
LOOP_ITERATIONS = 10  # number of seconds of data
BUFFER_SIZE = 5
ORDER_BOOK_DEPTH = 100
LOB_PATH = Path(
    "E:\\Quant_Projects\\microstructure-alpha-engine\\microstructure-alpha-engine\\data\\tester_folder\\"
)
TRADE_PATH = Path(
    "E:\\Quant_Projects\\microstructure-alpha-engine\\microstructure-alpha-engine\\data\\tester_folder\\"
)
client = Client(requests_params={"timeout": 5})

# Collect Data

In [ ]:
m_alpha_data.collect_data(
    SYMBOL, client, LOB_PATH, TRADE_PATH, LOOP_ITERATIONS, BUFFER_SIZE, ORDER_BOOK_DEPTH
)